# Chat Prompt Templates

The `chat.py` module defines prompt templates for constructing chat-model messages. It supports message placeholders, string-based messages, multimodal text, image, and dictionary content, chat-prompt composition, partial variables, synchronous and asynchronous formatting, and human-readable prompt representations.

# MessagePromptTemplateT: `TypeVar`

`MessagePromptTemplateT` represents a subtype of `BaseStringMessagePromptTemplate`.

```python
MessagePromptTemplateT = TypeVar(
    "MessagePromptTemplateT",
    bound="BaseStringMessagePromptTemplate" # Required parent type
)
```

# MessageLike: `TypeAlias`

`MessageLike` represents an object that can be used directly as a message inside a chat prompt.

```python
MessageLike = (
    BaseMessagePromptTemplate
    | BaseMessage
    | BaseChatPromptTemplate
)
```

# MessageLikeRepresentation: `TypeAlias`

`MessageLikeRepresentation` represents any supported message-template input format.

```python
MessageLikeRepresentation = (
    MessageLike
    | tuple[
        str | type,
        str | Sequence[dict[str, Any]] | Sequence[object]
    ]
    | str
    | dict[str, Any]
)
```


# Classes

1. `MessagesPlaceholder`:= Inserts a pre-existing list of messages into a chat prompt.

2. `BaseStringMessagePromptTemplate`:= Defines the abstract base for message templates backed by a string prompt.

3. `ChatMessagePromptTemplate`:= Creates a chat message with a custom role.

4. `HumanMessagePromptTemplate`:= Creates a message representing human input.

5. `AIMessagePromptTemplate`:= Creates a message representing AI output.

6. `SystemMessagePromptTemplate`:= Creates a system instruction message.

7. `BaseChatPromptTemplate`:= Defines the abstract base for chat prompt templates.

8. `ChatPromptTemplate`:= Builds and formats an ordered sequence of chat messages.

# MessagesPlaceholder: `BaseMessagePromptTemplate`

`MessagesPlaceholder` expects one input variable to contain a list of message representations.

The placeholder can be optional and can restrict the number of included messages to the most recent entries.

**Syntax**

```python
MessagesPlaceholder(
    variable_name: str, # Name of the variable containing messages
    *,
    optional: bool = False, # Whether the variable may be omitted
    **kwargs: Any # Additional model arguments
)
```

## Fields

1. `variable_name`:`str`:= Stores the name of the input variable containing the messages.

2. `optional`:`bool`:= Determines whether the placeholder may be omitted when formatting the prompt. Its default value is `False`.

3. `n_messages`:`PositiveInt | None`:= Specifies the maximum number of most-recent messages to include. Its default value is `None`.

## Properties

1. `input_variables`:`list[str]`:= Returns the placeholder variable when it is required or an empty list when it is optional.

## Methods

1. `__init__`:= Initializes a message placeholder using a variable name and optional behaviour.

   ```python
   __init__(
       self,
       variable_name: str, # Name of the variable containing messages
       *,
       optional: bool = False, # Whether the variable may be omitted
       **kwargs: Any # Additional model arguments
   ) -> None
   ```

2. `format_messages`:= Validates and converts the supplied value into `BaseMessage` objects.

   When `n_messages` is set, only the specified number of most-recent messages is retained.

   ```python
   format_messages(
       self,
       **kwargs: Any # Values used to resolve the placeholder
   ) -> list[BaseMessage]
   ```

3. `pretty_repr`:= Returns a human-readable representation of the placeholder.

   ```python
   pretty_repr(
       self,
       html: bool = False # Whether to use HTML-style formatting
   ) -> str
   ```

In [1]:
from langchain_core.prompts import MessagesPlaceholder


history = [
    ("human", "Hello"),
    ("ai", "Hi! How can I help?"),
    ("human", "Explain SQL joins.")
]


placeholder = MessagesPlaceholder(
    variable_name="history", # Variable containing message history
    optional=False, # History must be provided
    n_messages=2 # Keep only the two most-recent messages
)


messages = placeholder.format_messages(
    history=history # Value of the "history" variable
)


print("Variable name:", placeholder.variable_name)
print("Optional:", placeholder.optional)
print("Maximum messages:", placeholder.n_messages)
print("Input variables:", placeholder.input_variables)

print("\nFormatted messages:")

for message in messages:
    print(message.type, ":", message.content)

print("\nPretty representation:")
print(
    placeholder.pretty_repr(
        html=False # Use plain-text formatting
    )
)


optional_placeholder = MessagesPlaceholder(
    variable_name="history",
    optional=True # Variable may be omitted
)

print("\nOptional input variables:", optional_placeholder.input_variables)
print("Missing optional history:", optional_placeholder.format_messages())

Variable name: history
Optional: False
Maximum messages: 2
Input variables: ['history']

Formatted messages:
ai : Hi! How can I help?
human : Explain SQL joins.

Pretty representation:
============================= Messages Placeholder =============================

{history}

Optional input variables: []
Missing optional history: []



# BaseStringMessagePromptTemplate: `BaseMessagePromptTemplate`, `ABC`

`BaseStringMessagePromptTemplate` is an abstract base class for message prompt templates that use a `StringPromptTemplate`.

It provides factory methods, synchronous and asynchronous formatting, input-variable discovery, and readable representations.

## Fields
1. `prompt`:`StringPromptTemplate`:= Stores the string prompt template used to generate message content.
2. `additional_kwargs`:`dict[str, Any]`:= Stores additional fields passed to the generated message. Its default value is an empty dictionary.

## Properties
1. `input_variables`:`list[str]`:= Returns the input variables required by the stored string prompt.

## Methods
1. `from_template`:= Creates a message prompt template from a template string.
   ```python
   @classmethod
   from_template(
       cls,
       template: str, # Template string used to create the prompt
       template_format: PromptTemplateFormat = "f-string", # Template format
       partial_variables: dict[str, Any] | None = None, # Variables filled in advance
       **kwargs: Any # Additional constructor arguments
   ) -> Self
   ```
2. `from_template_file`:= Creates a message prompt template from a template file.
   ```python
   @classmethod
   from_template_file(
       cls,
       template_file: str | Path, # Path of the template file
       **kwargs: Any # Additional constructor arguments
   ) -> Self
   ```
3. `format`:= Abstract method that formats the stored prompt into one message.
   ```python
   format(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> BaseMessage
   ```
4. `aformat`:= Asynchronously formats the stored prompt into one message.
   ```python
   async aformat(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> BaseMessage
   ```
5. `format_messages`:= Synchronously formats the prompt and returns the resulting message inside a list.

   ```python
   format_messages(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> list[BaseMessage]
   ```
6. `aformat_messages`:= Asynchronously formats the prompt and returns the resulting message inside a list.
   ```python
   async aformat_messages(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> list[BaseMessage]
   ```
7. `pretty_repr`:= Returns a human-readable representation of the message prompt template.

   ```python
   pretty_repr(
       self,
       html: bool = False # Whether to use HTML-style formatting
   ) -> str
   ```

# ChatMessagePromptTemplate: `BaseStringMessagePromptTemplate`

`ChatMessagePromptTemplate` is a string-based message prompt template that produces a `ChatMessage` with a user-defined role.

## Fields

1. `role`:`str`:= Stores the role assigned to the generated chat message.

## Methods

1. `format`:= Synchronously formats the prompt and returns a `ChatMessage` with the configured role.

   ```python
   format(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> BaseMessage
   ```

2. `aformat`:= Asynchronously formats the prompt and returns a `ChatMessage` with the configured role.

   ```python
   async aformat(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> BaseMessage
   ```

In [2]:
#BaseStringMessagePromptTemplate is abstract, so this example uses its concrete subclass ChatMessagePromptTemplate.
from pathlib import Path

from langchain_core.prompts import ChatMessagePromptTemplate


# Create a template from a string
template = ChatMessagePromptTemplate.from_template(
    template="Hello {name}, welcome to {course}.", # Message template
    role="assistant", # Role of the generated ChatMessage
    additional_kwargs={"source": "demo"} # Extra message data
)


print("Prompt:", template.prompt) # Stored StringPromptTemplate
print("Additional kwargs:", template.additional_kwargs)
print("Input variables:", template.input_variables)


# Format into one ChatMessage
message = template.format(
    name="Saad", # Value for name
    course="LangChain" # Value for course
)

print("\nformat():", message)


# Format into a list containing one message
messages = template.format_messages(
    name="Saad",
    course="LangChain"
)

print("format_messages():", messages)


# Asynchronous formatting in Jupyter Notebook
async_message = await template.aformat(
    name="Saad",
    course="LangChain"
)

async_messages = await template.aformat_messages(
    name="Saad",
    course="LangChain"
)

print("aformat():", async_message)
print("aformat_messages():", async_messages)


# Human-readable representation
print("\npretty_repr():")
print(
    template.pretty_repr(
        html=False # Use plain-text formatting
    )
)


# Create a template from a file
Path("welcome_prompt.txt").write_text(
    "Welcome {name} to the {course} course."
)

file_template = ChatMessagePromptTemplate.from_template_file(
    template_file="welcome_prompt.txt", # Template-file path
    role="assistant" # Role of the generated message
)

print(
    "\nfrom_template_file():",
    file_template.format(
        name="Saad",
        course="Python"
    )
)

Prompt: input_variables=['course', 'name'] input_types={} partial_variables={} template='Hello {name}, welcome to {course}.'
Additional kwargs: {'source': 'demo'}
Input variables: ['course', 'name']

format(): content='Hello Saad, welcome to LangChain.' additional_kwargs={'source': 'demo'} response_metadata={} role='assistant'
format_messages(): [ChatMessage(content='Hello Saad, welcome to LangChain.', additional_kwargs={'source': 'demo'}, response_metadata={}, role='assistant')]
aformat(): content='Hello Saad, welcome to LangChain.' additional_kwargs={'source': 'demo'} response_metadata={} role='assistant'
aformat_messages(): [ChatMessage(content='Hello Saad, welcome to LangChain.', additional_kwargs={'source': 'demo'}, response_metadata={}, role='assistant')]

pretty_repr():
================================= Chat Message =================================

Hello {name}, welcome to {course}.

from_template_file(): content='Welcome Saad to the Python course.' additional_kwargs={} respon

# Shared Multimodal Message-Template Behaviour

`HumanMessagePromptTemplate`, `AIMessagePromptTemplate`, and `SystemMessagePromptTemplate` inherit common behaviour from the internal `_StringImageMessagePromptTemplate` class.

Their prompts may contain one string template or a list of text, image, and dictionary prompt templates.

## Inherited Fields

1. `prompt`:`StringPromptTemplate | list[StringPromptTemplate | ImagePromptTemplate | DictPromptTemplate]`:= Stores the template or templates used to generate message content.

2. `additional_kwargs`:`dict[str, Any]`:= Stores additional fields passed to the generated message. Its default value is an empty dictionary.

## Inherited Properties

1. `input_variables`:`list[str]`:= Returns all input variables required by the stored prompt templates.

## Inherited Methods

1. `from_template`:= Creates a message prompt from a string or a sequence of text, image, and dictionary templates.

   ```python
   @classmethod
   from_template(
       cls: type[Self],
       template: (
           str
           | Sequence[
               str
               | _TextTemplateParam
               | _ImageTemplateParam
               | dict[str, Any]
           ]
       ), # Template or multimodal template sequence
       template_format: PromptTemplateFormat = "f-string", # Template format
       *,
       partial_variables: dict[str, Any] | None = None, # Variables filled in advance
       **kwargs: Any # Additional constructor arguments
   ) -> Self
   ```

2. `from_template_file`:= Creates a message prompt template from the contents of a file.

   ```python
   @classmethod
   from_template_file(
       cls: type[Self],
       template_file: str | Path, # Path of the template file
       input_variables: list[str], # Variables expected by the template
       **kwargs: Any # Additional constructor arguments
   ) -> Self
   ```

3. `format_messages`:= Synchronously formats the template and returns the generated message inside a list.

   ```python
   format_messages(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> list[BaseMessage]
   ```

4. `aformat_messages`:= Asynchronously formats the template and returns the generated message inside a list.

   ```python
   async aformat_messages(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> list[BaseMessage]
   ```

5. `format`:= Synchronously formats string, image, and dictionary templates into one message.

   ```python
   format(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> BaseMessage
   ```

6. `aformat`:= Asynchronously formats string, image, and dictionary templates into one message.

   ```python
   async aformat(
       self,
       **kwargs: Any # Variables used to format the message
   ) -> BaseMessage
   ```

7. `pretty_repr`:= Returns a human-readable representation of every template used by the message prompt.

   ```python
   pretty_repr(
       self,
       html: bool = False # Whether to use HTML-style formatting
   ) -> str
   ```

In [4]:
from pathlib import Path

from langchain_core.prompts import HumanMessagePromptTemplate


# Create a multimodal template containing text, image, and dictionary content
multimodal_template = HumanMessagePromptTemplate.from_template(
    [
        {
            "type": "text",
            "text": "Describe {topic} using the given image."
        }, # Text template
        {
            "type": "image_url",
            "image_url": "https://example.com/{image_name}.png"
        }, # Image template
        {
            "type": "custom_data",
            "label": "{label}"
        } # Dictionary template
    ],
    additional_kwargs={
        "source": "multimodal-example"
    } # Additional message data
)


# Display inherited fields and property
print("Input variables:", multimodal_template.input_variables)
print("Additional kwargs:", multimodal_template.additional_kwargs)
print("Stored prompts:", multimodal_template.prompt)


# Format the template into one HumanMessage
message = multimodal_template.format(
    topic="network topology", # Text-template variable
    image_name="topology", # Image-template variable
    label="network diagram" # Dictionary-template variable
)

print("\nformat():")
print("Message type:", message.type)
print("Content:", message.content)
print("Additional kwargs:", message.additional_kwargs)


# Format the template into a list containing one HumanMessage
messages = multimodal_template.format_messages(
    topic="VLAN configuration",
    image_name="vlan",
    label="VLAN diagram"
)

print("\nformat_messages():")

for current_message in messages:
    print(current_message.type, ":", current_message.content)


# Format asynchronously
async_message = await multimodal_template.aformat(
    topic="router configuration", # Text-template variable
    image_name="router", # Image-template variable
    label="router diagram" # Dictionary-template variable
)

print("\naformat():")
print(async_message.content)


# Format asynchronously into a list
async_messages = await multimodal_template.aformat_messages(
    topic="switching",
    image_name="switch",
    label="switch diagram"
)

print("\naformat_messages():")

for current_message in async_messages:
    print(current_message.type, ":", current_message.content)


# Multimodal pretty_repr() is not fully supported for image templates
print("\nMultimodal pretty_repr():")

try:
    print(
        multimodal_template.pretty_repr(
            html=False # Use plain-text formatting
        )
    )
except NotImplementedError:
    print("pretty_repr() is not supported for templates containing images.")


# Use a text-only template to demonstrate pretty_repr()
text_template = HumanMessagePromptTemplate.from_template(
    "Explain {topic} simply." # Text-only template
)

print("\nText-only pretty_repr():")
print(
    text_template.pretty_repr(
        html=False # Use plain-text formatting
    )
)


# Create a message template from a file
Path("human_prompt.txt").write_text(
    "Explain {topic} in one sentence.",
    encoding="utf-8"
)

file_template = HumanMessagePromptTemplate.from_template_file(
    template_file="human_prompt.txt", # Path of the template file
    input_variables=["topic"] # Variables expected by the template
)

file_message = file_template.format(
    topic="inheritance" # Value used in the file template
)

print("\nfrom_template_file():")
print(file_message.content)

Input variables: ['topic', 'image_name', 'label']
Additional kwargs: {'source': 'multimodal-example'}
Stored prompts: [PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Describe {topic} using the given image.'), ImagePromptTemplate(input_variables=['image_name'], input_types={}, partial_variables={}, template={'url': 'https://example.com/{image_name}.png'}), DictPromptTemplate(template={'type': 'custom_data', 'label': '{label}'}, template_format='f-string')]

format():
Message type: human
Content: [{'type': 'text', 'text': 'Describe network topology using the given image.'}, {'type': 'image_url', 'image_url': {'url': 'https://example.com/topology.png'}}, {'type': 'custom_data', 'label': 'network diagram'}]
Additional kwargs: {'source': 'multimodal-example'}

format_messages():
human : [{'type': 'text', 'text': 'Describe VLAN configuration using the given image.'}, {'type': 'image_url', 'image_url': {'url': 'https://example.com/vlan.png'}}, {'type

# HumanMessagePromptTemplate: `_StringImageMessagePromptTemplate`

`HumanMessagePromptTemplate` is a multimodal message prompt template that creates `HumanMessage` objects representing user input.

**Syntax**
```python
HumanMessagePromptTemplate(
  self,
  *args: Any = (),
  **kwargs: Any = {}
)
```

In [5]:
from langchain_core.prompts import (
    HumanMessagePromptTemplate,
    PromptTemplate,
)


string_prompt = PromptTemplate.from_template(
    "Explain {topic} in a simple way." # String prompt template
)


human_template = HumanMessagePromptTemplate(
    prompt=string_prompt, # Prompt used to create the HumanMessage
    additional_kwargs={
        "source": "user-prompt"
    } # Additional data added to the generated message
)


message = human_template.format(
    topic="inheritance" # Value inserted into the prompt
)


print("Input variables:", human_template.input_variables)
print("Additional kwargs:", human_template.additional_kwargs)
print("Message type:", message.type)
print("Message content:", message.content)
print("Message additional kwargs:", message.additional_kwargs)

Input variables: ['topic']
Additional kwargs: {'source': 'user-prompt'}
Message type: human
Message content: Explain inheritance in a simple way.
Message additional kwargs: {'source': 'user-prompt'}


# AIMessagePromptTemplate: `_StringImageMessagePromptTemplate`
`AIMessagePromptTemplate` is a multimodal message prompt template that creates `AIMessage` objects representing AI-generated content.

**Syntax**
```python
AIMessagePromptTemplate(
  self,
  *args: Any = (),
  **kwargs: Any = {}
)
```



# SystemMessagePromptTemplate: `_StringImageMessagePromptTemplate`

`SystemMessagePromptTemplate` is a multimodal message prompt template that creates `SystemMessage` objects representing system-level instructions.
**Syntax**
```python
SystemMessagePromptTemplate(
  self,
  *args: Any = (),
  **kwargs: Any = {}
)
```

In [6]:
from langchain_core.prompts import PromptTemplate, SystemMessagePromptTemplate


prompt = PromptTemplate.from_template(
    "You are a {role} assistant." # System instruction template
)

system_template = SystemMessagePromptTemplate(
    prompt=prompt, # Prompt used to create the SystemMessage
    additional_kwargs={
        "source": "system-prompt"
    } # Additional message data
)

message = system_template.format(
    role="helpful Python" # Value inserted into the template
)


print("Input variables:", system_template.input_variables)
print("Message type:", message.type)
print("Message content:", message.content)
print("Additional kwargs:", message.additional_kwargs)

Input variables: ['role']
Message type: system
Message content: You are a helpful Python assistant.
Additional kwargs: {'source': 'system-prompt'}


# BaseChatPromptTemplate: `BasePromptTemplate[str]`, `ABC`

`BaseChatPromptTemplate` is an abstract prompt template for chat models.

It formats message templates into `ChatPromptValue` objects or their string representation and supports synchronous and asynchronous execution.

## Properties

1. `lc_attributes`:`dict[str, Any]`:= Returns the input variables that must be preserved during LangChain serialization.

## Methods

1. `format`:= Formats the complete chat prompt into a string.

   ```python
   format(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> str
   ```

2. `aformat`:= Asynchronously formats the complete chat prompt into a string.

   ```python
   async aformat(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> str
   ```

3. `format_prompt`:= Formats the chat prompt and wraps the messages in a `ChatPromptValue`.

   ```python
   format_prompt(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> ChatPromptValue
   ```

4. `aformat_prompt`:= Asynchronously formats the chat prompt and wraps the messages in a `ChatPromptValue`.

   ```python
   async aformat_prompt(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> ChatPromptValue
   ```

5. `format_messages`:= Abstract method that formats the prompt into a list of finalized messages.

   ```python
   format_messages(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> list[BaseMessage]
   ```

6. `aformat_messages`:= Asynchronously formats the prompt into a list of finalized messages.

   ```python
   async aformat_messages(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> list[BaseMessage]
   ```

7. `pretty_repr`:= Returns a human-readable representation of the chat prompt.

   Subclasses must provide the implementation.

   ```python
   pretty_repr(
       self,
       html: bool = False # Whether to use HTML-style formatting
   ) -> str
   ```

8. `pretty_print`:= Prints the human-readable prompt representation and enables HTML-style formatting in interactive environments.

   ```python
   pretty_print(
       self
   ) -> None
   ```

In [7]:
from langchain_core.prompts import ChatPromptTemplate


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a {role} assistant."),
        ("human", "Explain {topic} simply.")
    ]
)


# Property
print("LC attributes:", prompt.lc_attributes)


# 1. format()
formatted_text = prompt.format(
    role="Python", # System-message variable
    topic="inheritance" # Human-message variable
)

print("\nformat():")
print(formatted_text)


# 2. aformat()
async_formatted_text = await prompt.aformat(
    role="SQL", # System-message variable
    topic="joins" # Human-message variable
)

print("\naformat():")
print(async_formatted_text)


# 3. format_prompt()
prompt_value = prompt.format_prompt(
    role="Python",
    topic="polymorphism"
)

print("\nformat_prompt():")
print(type(prompt_value).__name__)
print(prompt_value.to_string())


# 4. aformat_prompt()
async_prompt_value = await prompt.aformat_prompt(
    role="Java",
    topic="encapsulation"
)

print("\naformat_prompt():")
print(type(async_prompt_value).__name__)
print(async_prompt_value.to_string())


# 5. format_messages()
messages = prompt.format_messages(
    role="Python",
    topic="classes"
)

print("\nformat_messages():")

for message in messages:
    print(message.type, ":", message.content)


# 6. aformat_messages()
async_messages = await prompt.aformat_messages(
    role="Python",
    topic="objects"
)

print("\naformat_messages():")

for message in async_messages:
    print(message.type, ":", message.content)


# 7. pretty_repr()
print("\npretty_repr():")
print(
    prompt.pretty_repr(
        html=False # Use plain-text formatting
    )
)


# 8. pretty_print()
print("\npretty_print():")
prompt.pretty_print()

LC attributes: {'input_variables': ['role', 'topic']}

format():
System: You are a Python assistant.
Human: Explain inheritance simply.

aformat():
System: You are a SQL assistant.
Human: Explain joins simply.

format_prompt():
ChatPromptValue
System: You are a Python assistant.
Human: Explain polymorphism simply.

aformat_prompt():
ChatPromptValue
System: You are a Java assistant.
Human: Explain encapsulation simply.

format_messages():
system : You are a Python assistant.
human : Explain classes simply.

aformat_messages():
system : You are a Python assistant.
human : Explain objects simply.

pretty_repr():
================================ System Message ================================

You are a {role} assistant.

================================ Human Message =================================

Explain {topic} simply.

pretty_print():
================================ System Message ================================

You are a {role} assistant.

================================ Human

# ChatPromptTemplate: `BaseChatPromptTemplate`

`ChatPromptTemplate` is a concrete chat prompt template that stores an ordered sequence of messages and message prompt templates.

It infers required and optional variables, supports composition and partial application, and formats every entry into a finalized chat message.

**Syntax**

```python
ChatPromptTemplate(
    messages: Sequence[MessageLikeRepresentation], # Messages used to build the prompt
    *,
    template_format: PromptTemplateFormat = "f-string", # Format used by string templates
    **kwargs: Any # Additional BasePromptTemplate arguments
)
```

## Fields

1. `messages`:`Annotated[list[MessageLike], SkipValidation()]`:= Stores the ordered messages and message prompt templates that form the chat prompt.

2. `validate_template`:`bool`:= Determines whether supplied input variables are checked against the variables inferred from the messages. Its default value is `False`.

## Methods

1. `__init__`:= Creates a chat prompt template from supported message representations.

   It automatically infers required, optional, and partial variables.

   ```python
   __init__(
       self,
       messages: Sequence[MessageLikeRepresentation], # Messages used to build the prompt
       *,
       template_format: PromptTemplateFormat = "f-string", # Format used by string templates
       **kwargs: Any # Additional BasePromptTemplate arguments
   ) -> None
   ```

2. `get_lc_namespace`:= Returns the LangChain serialization namespace for chat prompts.

   ```python
   @classmethod
   get_lc_namespace(
       cls # Chat-prompt class
   ) -> list[str]
   ```

3. `__add__`:= Combines the current chat prompt with another compatible prompt, message, message sequence, or string.

   ```python
   __add__(
       self,
       other: Any # Prompt or message content to append
   ) -> ChatPromptTemplate
   ```

4. `validate_input_variables`:= Infers input variables, optional variables, partial variables, and message-list input types.

   It can also validate explicitly supplied variables.

   ```python
   @classmethod
   validate_input_variables(
       cls,
       values: dict[str, Any] # Model values being validated
   ) -> Any
   ```

5. `from_template`:= Creates a chat prompt containing one human message generated from a string template.

   ```python
   @classmethod
   from_template(
       cls,
       template: str, # Template used for the human message
       **kwargs: Any # Arguments used to create the prompt template
   ) -> ChatPromptTemplate
   ```

6. `from_messages`:= Creates a chat prompt from a sequence of supported message representations.

   ```python
   @classmethod
   from_messages(
       cls,
       messages: Sequence[MessageLikeRepresentation], # Message representations to convert
       template_format: PromptTemplateFormat = "f-string" # Format used by string templates
   ) -> ChatPromptTemplate
   ```

7. `format_messages`:= Synchronously formats every message template and returns a list of finalized messages.

   ```python
   format_messages(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> list[BaseMessage]
   ```

8. `aformat_messages`:= Asynchronously formats every message template and returns a list of finalized messages.

   ```python
   async aformat_messages(
       self,
       **kwargs: Any # Variables used to format all messages
   ) -> list[BaseMessage]
   ```

9. `partial`:= Creates a new chat prompt template with selected variables already assigned.

   ```python
   partial(
       self,
       **kwargs: Any # Input variables and values to fill in advance
   ) -> ChatPromptTemplate
   ```

10. `append`:= Adds one supported message representation to the end of the chat prompt.

    ```python
    append(
        self,
        message: MessageLikeRepresentation # Message representation to append
    ) -> None
    ```

11. `extend`:= Adds multiple supported message representations to the end of the chat prompt.

    ```python
    extend(
        self,
        messages: Sequence[MessageLikeRepresentation] # Message representations to append
    ) -> None
    ```

12. `__getitem__`:= Returns one message for an integer index or a new `ChatPromptTemplate` for a slice.

    ```python
    __getitem__(
        self,
        index: int | slice # Message index or slice
    ) -> MessageLike | ChatPromptTemplate
    ```

13. `__len__`:= Returns the number of messages stored in the chat prompt.

    ```python
    __len__(
        self
    ) -> int
    ```

14. `save`:= Represents the deprecated prompt-saving interface.

    This implementation raises `NotImplementedError`. Use LangChain load and dump utilities instead.

    ```python
    save(
        self,
        file_path: Path | str # Intended destination path
    ) -> None
    ```

15. `pretty_repr`:= Returns a human-readable representation by joining the representations of all stored messages.

    ```python
    pretty_repr(
        self,
        html: bool = False # Whether to use HTML-style formatting
    ) -> str
    ```

In [8]:
import warnings

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


def display_messages(title, messages):
    """Display formatted messages clearly."""
    print(f"\n{title}")

    for message in messages:
        print(f"{message.type}: {message.content}")


# 1. __init__(): Create a ChatPromptTemplate directly
prompt = ChatPromptTemplate(
    messages=[
        ("system", "You are a {role} assistant."),
        MessagesPlaceholder(
            variable_name="history", # Variable containing previous messages
            optional=True # History may be omitted
        ),
        ("human", "Explain {topic} in a {style} way.")
    ], # Ordered message representations
    template_format="f-string", # Template formatting style
    input_variables=["role", "style", "topic"], # Explicit required variables
    validate_template=True # Validate supplied variables
)


# Fields
print("Stored messages:", len(prompt.messages))
print("Validate template:", prompt.validate_template)
print("Required variables:", prompt.input_variables)
print("Optional variables:", prompt.optional_variables)

print("\nStored message templates:")

for message_template in prompt.messages:
    print(type(message_template).__name__)


# 2. get_lc_namespace()
namespace = ChatPromptTemplate.get_lc_namespace()

print("\nLangChain namespace:", namespace)


# 3. __add__(): Combine two chat prompts
additional_prompt = ChatPromptTemplate.from_messages(
    [
        ("ai", "Important point: {key_point}")
    ] # Message added to the original prompt
)

combined_prompt = prompt + additional_prompt

combined_messages = combined_prompt.format_messages(
    role="Python", # System-message variable
    topic="inheritance", # Human-message variable
    style="simple", # Human-message variable
    key_point="A child class can reuse parent-class features." # Added variable
)

display_messages("__add__():", combined_messages)


# 4. validate_input_variables()
# This validator normally runs automatically during construction.
validated_values = ChatPromptTemplate.validate_input_variables(
    {
        "messages": prompt.messages, # Converted message templates
        "input_variables": ["role", "style", "topic"], # Expected variables
        "validate_template": True # Enable variable validation
    }
)

print(
    "\nValidated input variables:",
    validated_values["input_variables"]
)

print(
    "Inferred optional variables:",
    validated_values["optional_variables"]
)


# Demonstrate invalid input-variable validation
try:
    ChatPromptTemplate.validate_input_variables(
        {
            "messages": prompt.messages, # Message templates to inspect
            "input_variables": ["topic"], # Incorrect variable list
            "validate_template": True # Enable validation
        }
    )

except ValueError as error:
    print("\nValidation error:", error)


# 5. from_template(): Creates one human-message template
single_message_prompt = ChatPromptTemplate.from_template(
    "Give one example of {topic}." # Human-message template
)

single_messages = single_message_prompt.format_messages(
    topic="polymorphism" # Template variable
)

display_messages("from_template():", single_messages)


# 6. from_messages(): Creates a prompt from multiple representations
factory_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Respond using {language}."),
        ("human", "Define {concept}.")
    ], # Message representations
    template_format="f-string" # Template formatting style
)

factory_messages = factory_prompt.format_messages(
    language="simple English",
    concept="encapsulation"
)

display_messages("from_messages():", factory_messages)


# 7. format_messages(): Synchronous formatting
formatted_messages = prompt.format_messages(
    role="programming", # System-message variable
    history=[
        ("human", "What is OOP?"),
        ("ai", "OOP organizes programs around objects.")
    ], # Optional message history
    topic="inheritance", # Human-message variable
    style="beginner-friendly" # Human-message variable
)

display_messages("format_messages():", formatted_messages)


# 8. aformat_messages(): Asynchronous formatting
async_messages = await prompt.aformat_messages(
    role="database", # System-message variable
    topic="SQL joins", # Human-message variable
    style="concise" # Human-message variable
)

display_messages("aformat_messages():", async_messages)


# 9. partial(): Assign selected variables in advance
partial_prompt = prompt.partial(
    role="Python", # Preassigned role
    style="simple" # Preassigned explanation style
)

print(
    "\nVariables remaining after partial():",
    partial_prompt.input_variables
)

partial_messages = partial_prompt.format_messages(
    topic="decorators" # Only remaining required variable
)

display_messages("partial():", partial_messages)


# Create a separate prompt for mutation methods
editable_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Answer clearly.")
    ]
)


# 10. append(): Add one message
editable_prompt.append(
    ("human", "Explain {topic}.") # Message appended at the end
)

print(
    "\nLength after append():",
    len(editable_prompt)
)


# 11. extend(): Add multiple messages
editable_prompt.extend(
    [
        ("ai", "I will explain {topic}."),
        ("human", "Also mention {example}.")
    ] # Messages appended at the end
)

print(
    "Length after extend():",
    len(editable_prompt)
)

editable_messages = editable_prompt.format_messages(
    topic="classes",
    example="a Student class"
)

display_messages("append() and extend():", editable_messages)


# 12. __getitem__(): Access one message
first_message_template = editable_prompt[0]

print(
    "\nFirst stored item:",
    type(first_message_template).__name__
)


# A slice returns another ChatPromptTemplate
sliced_prompt = editable_prompt[1:3]

print(
    "Sliced prompt type:",
    type(sliced_prompt).__name__
)

print(
    "Messages in sliced prompt:",
    len(sliced_prompt)
)


# 13. __len__(): Return the number of stored messages
print(
    "\nTotal messages:",
    len(editable_prompt)
)


# 14. save(): Current implementation raises NotImplementedError
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        editable_prompt.save(
            "chat_prompt.json" # Intended destination file
        )

except NotImplementedError:
    print(
        "\nsave(): ChatPromptTemplate.save() is not implemented."
    )


# 15. pretty_repr(): Display a readable prompt representation
print("\npretty_repr():")

print(
    prompt.pretty_repr(
        html=False # Use plain-text representation
    )
)

Stored messages: 3
Validate template: True
Required variables: ['role', 'style', 'topic']
Optional variables: ['history']

Stored message templates:
SystemMessagePromptTemplate
MessagesPlaceholder
HumanMessagePromptTemplate

LangChain namespace: ['langchain', 'prompts', 'chat']

__add__():
system: You are a Python assistant.
human: Explain inheritance in a simple way.
ai: Important point: A child class can reuse parent-class features.

Validated input variables: ['role', 'style', 'topic']
Inferred optional variables: ['history']

Validation error: Got mismatched input_variables. Expected: {'style', 'role', 'topic'}. Got: ['topic']

from_template():
human: Give one example of polymorphism.

from_messages():
system: Respond using simple English.
human: Define encapsulation.

format_messages():
system: You are a programming assistant.
human: What is OOP?
ai: OOP organizes programs around objects.
human: Explain inheritance in a beginner-friendly way.

aformat_messages():
system: You are a